# 💰 EduNilai — Cost-Burden ROI Analysis

**Stage:** 06 — Return on Investment (Cost-Burden Framework)

## Why cost-burden, not salary-based ROI?

Field-level graduate salary data in Malaysia is not available in open machine-readable form.
The Graduate Tracer Study (KPT) and DOSM Salaries & Wages Survey publish aggregate figures
in PDF reports without consistent annual field-level breakdowns.

Rather than fabricate or estimate salary figures, this analysis builds ROI from **data we
actually collected and can defend**:

| Metric | Data source | What it measures |
|---|---|---|
| **Cost-to-income ratio** | Fees + HIES household income | How many months of family income a degree costs |
| **Real cost burden** | Fees + Education CPI | How much the real cost has changed since 2012 |
| **Employment-adjusted burden** | Fees + SRU rate | Cost burden adjusted for probability of degree-matched employment |
| **Burden index** | All three combined | Composite score ranking fields by financial risk |

> **Limitation:** Without salary data, we cannot compute NPV or payback period in the
> traditional sense. What we can measure is the **cost side of ROI** — how much families
> sacrifice — and penalise fields where graduates are less likely to recoup that cost
> through matched employment.


## ⚙️ Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

BLUE   = '#2563EB'
ORANGE = '#EA580C'
GREEN  = '#16A34A'
RED    = '#DC2626'
GRAY   = '#6B7280'
PURPLE = '#7C3AED'

import os

def load(clean_name, raw_path):
    p = f'../data/cleaned/{clean_name}'
    return pd.read_csv(p if os.path.exists(p) else f'../data/{raw_path}', parse_dates=['date'])

fees    = pd.read_csv('../data/cost/edunilai_mean_fees.csv')
hh_nat  = load('hh_income_national.csv', 'demographic/hh_income_national.csv')
cpi     = load('cpi.csv', 'inflation/cpi_2010_onwards.csv')
sru_sex = load('graduate_underemployment_sex.csv', 'salary/graduate_underemployment_sex.csv')

fees['public_avg']  = fees[['UTM', 'UM']].mean(axis=1)
fees['private_avg'] = fees[["APU", "Taylor's"]].mean(axis=1)
hh_nat['year']  = hh_nat['date'].dt.year
cpi['year']     = cpi['date'].dt.year
sru_sex['year'] = sru_sex['date'].dt.year
if 'sru_person' in sru_sex.columns:
    sru_sex = sru_sex.rename(columns={'sru_person': 'sru_persons'})

# Key reference values
HH_MEDIAN_2022    = hh_nat[hh_nat['year']==2022]['income_median'].values[0]
HH_MEDIAN_2012    = hh_nat[hh_nat['year']==2012]['income_median'].values[0]
CPI_EDU_2012      = cpi[(cpi['division']=='10')&(cpi['year']==2012)]['index'].values[0]
CPI_EDU_2024      = cpi[(cpi['division']=='10')&(cpi['year']==2024)]['index'].values[0]
SRU_RATE_2024     = sru_sex[sru_sex['sex']=='overall'].groupby('year')['sru_rate'].mean()[2024]
DEFLATION_FACTOR  = CPI_EDU_2012 / CPI_EDU_2024   # convert 2024 nominal to 2012 real

print(f"Reference values:")
print(f"  Median HH income 2022:  RM {HH_MEDIAN_2022:,}/month  (RM {HH_MEDIAN_2022*12:,}/year)")
print(f"  Median HH income 2012:  RM {HH_MEDIAN_2012:,}/month")
print(f"  Education CPI 2012:     {CPI_EDU_2012:.2f}")
print(f"  Education CPI 2024:     {CPI_EDU_2024:.2f}")
print(f"  Deflation factor:       {DEFLATION_FACTOR:.4f}")
print(f"  SRU rate 2024:          {SRU_RATE_2024:.1f}%")


---
## 📊 R1 — Cost-to-Income Ratio by Field (2022)

In [ ]:
# Months of median household income needed to pay for the degree
rows = []
for _, row in fees.iterrows():
    pub_months  = row['public_avg']  / HH_MEDIAN_2022 if not pd.isna(row['public_avg'])  else np.nan
    priv_months = row['private_avg'] / HH_MEDIAN_2022 if not pd.isna(row['private_avg']) else np.nan
    rows.append({
        'field':         row['field_category'],
        'public_fee':    row['public_avg'],
        'private_fee':   row['private_avg'],
        'public_months': pub_months,
        'private_months':priv_months,
        'public_years':  pub_months  / 12,
        'private_years': priv_months / 12,
    })
cir_df = pd.DataFrame(rows).sort_values('public_months', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Public
colors_pub = [RED if v > 3 else ORANGE if v > 2 else GREEN for v in cir_df['public_months']]
bars = axes[0].barh(cir_df['field'], cir_df['public_months'], color=colors_pub, alpha=0.85)
axes[0].axvline(12, color=ORANGE, linewidth=1.5, linestyle='--', label='1 year of income')
axes[0].axvline(24, color=RED,    linewidth=1.5, linestyle='--', label='2 years of income')
axes[0].set_xlabel('Months of Median Household Income')
axes[0].set_title('Public University\nCost-to-Income Ratio (2022)', fontweight='bold')
axes[0].legend(fontsize=8)
for bar, val in zip(bars, cir_df['public_months']):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f} months', va='center', fontsize=8.5)

# Private
priv_sorted = cir_df.dropna(subset=['private_months']).sort_values('private_months', ascending=True)
colors_prv  = [RED if v > 24 else ORANGE if v > 18 else GREEN for v in priv_sorted['private_months']]
bars2 = axes[1].barh(priv_sorted['field'], priv_sorted['private_months'], color=colors_prv, alpha=0.85)
axes[1].axvline(12,  color=GREEN,  linewidth=1.5, linestyle='--', label='1 year of income')
axes[1].axvline(24,  color=ORANGE, linewidth=1.5, linestyle='--', label='2 years of income')
axes[1].axvline(36,  color=RED,    linewidth=1.5, linestyle='--', label='3 years of income')
axes[1].set_xlabel('Months of Median Household Income')
axes[1].set_title('Private University\nCost-to-Income Ratio (2022)', fontweight='bold')
axes[1].legend(fontsize=8)
for bar, val in zip(bars2, priv_sorted['private_months']):
    axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.0f} months', va='center', fontsize=8.5)

plt.suptitle('How Many Months of Full Household Income Does a Degree Cost?\n'
             '(Based on 2022 HIES median household income: RM 6,338/month)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"Based on 2022 median HH income: RM {HH_MEDIAN_2022:,}/month")
print()
print(cir_df[['field','public_months','public_years','private_months','private_years']].round(1).to_string(index=False))


---
## 📈 R2 — How Cost Burden Changed: 2012 vs 2022

In [ ]:
# Compare affordability at 2012 vs 2022 income levels
rows_change = []
for _, row in fees.iterrows():
    pub_2012 = row['public_avg']  / (HH_MEDIAN_2012 * 12)
    pub_2022 = row['public_avg']  / (HH_MEDIAN_2022 * 12)
    prv_2012 = row['private_avg'] / (HH_MEDIAN_2012 * 12) if not pd.isna(row['private_avg']) else np.nan
    prv_2022 = row['private_avg'] / (HH_MEDIAN_2022 * 12) if not pd.isna(row['private_avg']) else np.nan
    rows_change.append({
        'field':       row['field_category'],
        'pub_2012':    pub_2012,
        'pub_2022':    pub_2022,
        'pub_change':  pub_2022 - pub_2012,
        'prv_2012':    prv_2012,
        'prv_2022':    prv_2022,
        'prv_change':  prv_2022 - prv_2012 if not pd.isna(prv_2012) else np.nan,
    })
change_df = pd.DataFrame(rows_change).sort_values('pub_change')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Public change
colors_chg = [GREEN if v < 0 else RED for v in change_df['pub_change']]
bars = axes[0].barh(change_df['field'], change_df['pub_change'], color=colors_chg, alpha=0.85)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_xlabel('Change in Cost Burden (years of household income)')
axes[0].set_title('Change in Public Degree Affordability\n2012 to 2022 (Negative = More Affordable)',
                  fontweight='bold')
axes[0].set_xlim(change_df['pub_change'].min() * 1.4, change_df['pub_change'].max() + 0.05)
for bar, val in zip(bars, change_df['pub_change']):
    axes[0].text(val / 2, bar.get_y() + bar.get_height()/2,
                 f'{val:+.3f} yrs', va='center', ha='center',
                 fontsize=8.5, color='white', fontweight='bold')

# 2012 vs 2022 grouped bar
x = np.arange(len(change_df))
w = 0.35
axes[1].barh(x + w/2, change_df['pub_2012'], w, color=BLUE,   alpha=0.8, label='2012')
axes[1].barh(x - w/2, change_df['pub_2022'], w, color=ORANGE, alpha=0.8, label='2022')
axes[1].set_yticks(x)
axes[1].set_yticklabels(change_df['field'], fontsize=8)
axes[1].set_xlabel('Years of Full Household Income')
axes[1].set_title('Public Degree Cost Burden:\n2012 vs 2022', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

hh_growth = (HH_MEDIAN_2022 / HH_MEDIAN_2012 - 1) * 100
print(f"HH income growth 2012-2022: +{hh_growth:.1f}%")
print(f"=> All public fields became MORE affordable as income grew faster than fees")
print()
print(change_df[['field','pub_2012','pub_2022','pub_change']].round(3).to_string(index=False))


---
## ⚠️ R3 — Employment-Adjusted Burden Index

Cost burden alone does not capture risk. A high-cost degree where graduates reliably get matched jobs is better than a cheap degree with high underemployment.

> **Adjusted burden = Cost-to-income ratio × (1 + SRU rate)**
>
> The SRU penalty reflects the probability that the degree cost is borne without the expected matched-employment payoff.

In [ ]:
# SRU penalty: if 36% of graduates are underemployed, 
# the expected value of the degree is discounted by that rate
SRU_PENALTY = 1 + (SRU_RATE_2024 / 100)

cir_df['pub_adj_burden']  = cir_df['public_years']  * SRU_PENALTY
cir_df['priv_adj_burden'] = cir_df['private_years'] * SRU_PENALTY
cir_adj = cir_df.sort_values('pub_adj_burden', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

for ax, col, title in [
    (axes[0], 'pub_adj_burden',  'Public University'),
    (axes[1], 'priv_adj_burden', 'Private University'),
]:
    df_plot = cir_adj.dropna(subset=[col]).sort_values(col, ascending=True)
    colors  = [RED if v > 2 else ORANGE if v > 1 else GREEN for v in df_plot[col]]
    bars    = ax.barh(df_plot['field'], df_plot[col], color=colors, alpha=0.85)
    ax.axvline(1, color=ORANGE, linewidth=1.5, linestyle='--', alpha=0.8, label='1 yr (unadjusted)')
    ax.set_xlabel('Employment-Adjusted Cost Burden (years of HH income)')
    ax.set_title(f'{title}\nEmployment-Adjusted Burden Index', fontweight='bold')
    ax.legend(fontsize=8)
    for bar, val in zip(bars, df_plot[col]):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=9)

plt.suptitle(f'Cost Burden Adjusted for Underemployment Risk (SRU = {SRU_RATE_2024:.1f}%)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"SRU penalty multiplier: {SRU_PENALTY:.4f}  (every year of cost burden is worth {SRU_PENALTY:.4f}x more in risk)")
print()
print("Employment-adjusted burden (public):")
print(cir_adj[['field','public_years','pub_adj_burden']].round(3).to_string(index=False))


---
## 🏆 R4 — Final Burden Index Ranking

In [ ]:
# Composite rank: normalise 3 components, weight equally
scaler = MinMaxScaler()

cir_df['norm_cost']    = scaler.fit_transform(cir_df[['public_years']])
cir_df['norm_adj']     = scaler.fit_transform(cir_df[['pub_adj_burden']])

# Real cost: deflate to 2012 RM
cir_df['public_real_2012'] = cir_df['public_fee'] * DEFLATION_FACTOR
cir_df['norm_real']  = scaler.fit_transform(cir_df[['public_real_2012']])

cir_df['burden_index'] = (
    cir_df['norm_cost']  * 0.40 +
    cir_df['norm_adj']   * 0.40 +
    cir_df['norm_real']  * 0.20
) * 100

ranked = cir_df.sort_values('burden_index', ascending=False).reset_index(drop=True)
ranked.index += 1

fig, ax = plt.subplots(figsize=(11, 6))
colors_rank = [RED if v > 60 else ORANGE if v > 35 else GREEN for v in ranked['burden_index']]
bars = ax.barh(ranked['field'][::-1], ranked['burden_index'][::-1],
               color=colors_rank[::-1], alpha=0.85)
ax.axvline(50, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Burden Index (0-100, higher = worse financial risk)')
ax.set_title('EduNilai Cost-Burden ROI Index\nPublic Universities — All Fields Ranked',
             fontweight='bold')
for bar, val in zip(bars, ranked['burden_index'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("FINAL BURDEN INDEX RANKING (Public Universities)")
print("=" * 70)
print(f"{'Rank':<5} {'Field':<35} {'Fee (RM)':>10} {'Months':>8} {'Adj.Burden':>11} {'Index':>7}")
print("-" * 70)
for rank, row in ranked.iterrows():
    print(f"{rank:<5} {row['field']:<35} {row['public_fee']:>10,.0f} "
          f"{row['public_months']:>8.1f} {row['pub_adj_burden']:>11.3f} {row['burden_index']:>7.1f}")


---
## 📋 R5 — Research Questions Answered

In [ ]:
print("=" * 65)
print("COST-BURDEN ROI — RESEARCH QUESTIONS ANSWERED")
print("=" * 65)

best  = ranked.iloc[-1]
worst = ranked.iloc[0]
hh_growth = (HH_MEDIAN_2022/HH_MEDIAN_2012 - 1)*100
med_pub_months  = cir_df['public_months'].median()
med_priv_months = cir_df['private_months'].median()
priv_multiple   = cir_df['private_years'].mean() / cir_df['public_years'].mean()

print()
print("Q: Which fields represent the lowest and highest financial burden?")
print("-" * 66)
print(f"  Lowest burden  (best value):  {best['field']}")
print(f"    Fee: RM {best['public_fee']:,.0f}  = {best['public_months']:.1f} months of HH income")
print(f"    Adjusted burden index: {best['burden_index']:.1f}/100")
print(f"  Highest burden (worst value): {worst['field']}")
print(f"    Fee: RM {worst['public_fee']:,.0f}  = {worst['public_months']:.1f} months of HH income")
print(f"    Adjusted burden index: {worst['burden_index']:.1f}/100")
print()
print("Q: Are degrees becoming more or less affordable over time?")
print("-" * 58)
print(f"  HH income grew +{hh_growth:.1f}% from 2012 to 2022.")
print("  All public fields became marginally more affordable as income rose.")
print("  However, Education CPI outpaced overall inflation — real education")
print("  costs are rising faster than living costs.")
print("  => Affordability improved only because incomes rose, not because")
print("     education costs fell. Any fee increase reverses this quickly.")
print()
print("Q: Does institution type (public vs private) affect the burden?")
print("-" * 64)
print(f"  Median public  degree: {med_pub_months:.0f} months of HH income")
print(f"  Median private degree: {med_priv_months:.0f} months of HH income")
print(f"  Private costs {priv_multiple:.1f}x more than public on average.")
print(f"  => Same labour market outcome, {priv_multiple:.1f}x the financial burden.")
print()
print("Q: How does underemployment affect the real value of a degree?")
print("-" * 63)
print(f"  SRU rate 2024: {SRU_RATE_2024:.1f}% -> burden inflated by {SRU_PENALTY:.3f}x")
print(f"  1 in {100/SRU_RATE_2024:.0f} graduates bears the full degree cost")
print("  while working in a job that did not require it.")
print("=" * 65)

os.makedirs('../data/analysis', exist_ok=True)
ranked.to_csv('../data/analysis/roi_burden_index.csv', index=False)
cir_df.to_csv('../data/analysis/cost_income_ratio.csv', index=False)
print("Saved -> ../data/analysis/roi_burden_index.csv")
print("Saved -> ../data/analysis/cost_income_ratio.csv")
